# 45 · Frozen replay with sensor-only eligibility

Replay the five saved folds of the seed-42 fixed-lag motion model using the MPS backend.
Use the cached sensor preparation from [notebook 44](44_label_free_sensor_audit.ipynb).
The replay compares histories built from sensor eligibility and sensor segments against the prepared reference.
There is no assumed direction or size of any score change.

The candidate uses only the seven sensor channels. It applies the saved motion mean, scaler, and
window transformation from each fold without fitting. It predicts before joining development labels
for scoring. The final-test participants remain reserved.

A numerical control must reproduce the historical reference probabilities before interpreting
candidate scores. This check is independent of candidate accuracy. The replay uses frozen research
checkpoints and leaves the deployment bundle's exact-runtime gate in place.

## Locate the checkout and set run options

**What:** Resolve the project root and set the setup and replay flags.

**Why:** With both flags off, the notebook reads the plan and displays verified results if present.
Set `SLEEP_CLASSIFIER_ROOT` to use an explicit checkout path.

In [ ]:
import logging
import os
import shutil
import sys
from pathlib import Path

from IPython import get_ipython
from IPython.display import display

logging.basicConfig(level=logging.INFO, stream=sys.stdout, force=True)
logger = logging.getLogger("sensor_only_replay_notebook")
ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "pyproject.toml").is_file() and (path / "src/sleep_classifier").is_dir()),
    Path.cwd() / "sleep-classifier",
)
ROOT = Path(os.environ.get("SLEEP_CLASSIFIER_ROOT", str(ROOT))).expanduser().resolve()

REPOSITORY_URL = "https://github.com/rbl-bidilatimotei/sleep-classifier.git"
BRANCH = "feature/sensor-only-frozen-replay"
RUN_SETUP = False
RUN_REPLAY = False

logger.info("Repository=%s setup=%s replay=%s", ROOT, RUN_SETUP, RUN_REPLAY)

## Set up the environment when requested

**What:** Clone the feature branch if the checkout is missing. Sync the locked GRU, notebook, and
development groups.

**Why:** Inexact sync preserves other installed groups. Set `RUN_SETUP = True` and run the flag
and setup cells. Then select the Python kernel from the checkout's `.venv` if necessary.
Restore `RUN_SETUP = False` before saving the source notebook.

The new checkout also needs the completed audit snapshot and saved research artifacts specified
in the replay configuration. Keep their original paths relative to the checkout.

In [ ]:
if RUN_SETUP:
    if not ROOT.exists():
        !git clone --branch {BRANCH} {REPOSITORY_URL} "{ROOT}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("Repository clone failed")
    if shutil.which("uv") is None:
        installer = ROOT / "artifacts/uv-install.sh"
        installer.parent.mkdir(parents=True, exist_ok=True)
        !curl -LsSf https://astral.sh/uv/install.sh -o "{installer}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("uv download failed")
        !sh "{installer}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("uv installation failed")
        os.environ["PATH"] = str(Path.home() / ".local/bin") + os.pathsep + os.environ["PATH"]
    %cd "{ROOT}"
    !uv sync --locked --inexact --group gru --group notebook --group dev
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Environment setup failed")
else:
    logger.info("Environment setup skipped")


## Inspect the replay plan

**What:** Verify the configured checkpoint, audit, and fold provenance. Show the resolved paths.

**Why:** The metadata identifies the reference and validation scope before model inference starts.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from sleep_classifier.sensor_only_replay import SensorOnlyReplayStudy

study = SensorOnlyReplayStudy(ROOT / "configs/sensor-only-replay.yaml")
plan = study.plan()
logger.info("Replay config=%s output=%s", study.config_path, study.output_dir)
display(plan)


## Run the frozen replay when requested

**What:** Validate readiness and run the CLI only when the flag is enabled and the output path is free.

**Why:** The CLI runs frozen inference. An existing output is checked by the next cell and kept intact.

To start inference, set `RUN_REPLAY = True` and run the flag, plan, and replay cells. Restore
`RUN_REPLAY = False` after the run. Keep both flags false in the saved notebook.
From the project root, the equivalent command is:

```bash
uv run --locked --inexact --group gru --group notebook python -u \
  -m sleep_classifier.sensor_only_replay --config configs/sensor-only-replay.yaml --run
```

Use a new config and output path for an intentional repeat. The command performs no training.

In [ ]:
if RUN_REPLAY and not study.output_dir.exists():
    study.validate_run_start()
    logger.info("Run replay config=%s output=%s", study.config_path, study.output_dir)
    %cd "{ROOT}"
    !uv run --locked --inexact --group gru --group notebook python -u \
        -m sleep_classifier.sensor_only_replay --config "{study.config_path}" --run
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Frozen sensor replay failed")
elif study.output_dir.exists():
    logger.info("Verify the existing replay at %s", study.output_dir)
else:
    logger.info("Replay skipped because RUN_REPLAY is disabled")


## Load the replay status

**What:** Read the report and verify its provenance and retained results.

**Why:** The following tables require a complete, verified report. Pending or partial results
stay visible as status messages.

In [ ]:
report = study.read_report()
report_ready = report["status"] == "passed"
logger.info("Replay status=%s output=%s", report["status"], study.output_dir)
if report["status"] == "pending":
    logger.info("Enable RUN_REPLAY to create the report")
elif report["status"] == "partial":
    logger.info("The output directory exists but its report is incomplete")


## Check numerical control and integrity

**What:** Show each fold's reference control and integrity checks.

**Why:** The historical probability control tests the restored inference path. Candidate accuracy
cannot compensate for a failed control.

In [ ]:
if report["folds"]:
    checks = [
        {"fold": fold["fold"], "control": fold["control"], "integrity": fold["integrity"]}
        for fold in report["folds"]
    ]
    display(pd.json_normalize(checks).set_index("fold").T.rename_axis("check"))


## Compare fold scope and paired scores

**What:** Count epochs in each fold and compare accuracy, macro F1, and validation NLL on shared scored
epochs. NLL is negative log likelihood.

**Why:** Both predictors must use the same scored identities for paired metrics. The candidate can
predict additional unscored epochs, which contribute no score. The conservative sensor clock
excludes three reference targets at the session tail. Those targets also contribute no paired score.

In [ ]:
SCOPE_COLUMNS = (
    "candidate_epochs", "reference_epochs", "shared_scored_epochs",
    "candidate_only_epochs", "reference_only_epochs",
)
METRIC_COLUMNS = ("accuracy", "macro_f1", "validation_nll")

if report_ready:
    comparisons = {fold["fold"]: fold["comparison"] for fold in report["folds"]}
    scope_rows = [
        {"fold": fold, **{key: comparison["scopes"][key] for key in SCOPE_COLUMNS}}
        for fold, comparison in comparisons.items()
    ]
    score_rows = [
        {"fold": fold, "predictor": predictor,
         **{key: comparison[predictor][key] for key in METRIC_COLUMNS}}
        for fold, comparison in comparisons.items()
        for predictor in ("reference", "candidate")
    ]
    display(pd.DataFrame(scope_rows).set_index("fold"))
    display(pd.DataFrame(score_rows).set_index(["fold", "predictor"]).round(6))


## Review pooled scores and reference-only targets

**What:** Show paired scores and class metrics across folds. List reference-only identities,
then show the reference scores on all of its eligible targets.

**Why:** The paired table uses shared scored epochs. The separate reference population shows
the effect of changing the scoring scope.

In [ ]:
STAGES = ("Wake", "Light", "Deep", "REM")

if report_ready:
    pooled = report["pooled"]
    display(pd.DataFrame(
        {key: pooled["scopes"][key] for key in SCOPE_COLUMNS}, index=["Pooled epochs"],
    ))
    display(pd.DataFrame(
        [pooled["reference"], pooled["candidate"]], index=["Reference", "Sensor replay"],
    )[list(METRIC_COLUMNS)].rename_axis("Shared scored epochs").round(6))

    class_rows = [
        {"stage": stage, "predictor": predictor, **pooled[predictor]["classes"][str(index)]}
        for index, stage in enumerate(STAGES)
        for predictor in ("reference", "candidate")
    ]
    display(pd.DataFrame(class_rows).set_index(["stage", "predictor"]).round(6))

    display(pd.DataFrame(pooled["reference_only_identities"]))
    logger.info("%s", pooled["validation_nll_definition"])
    display(pd.DataFrame(
        [pooled["reference_all_eligible"]], index=["Reference on all eligible targets"],
    )[list(METRIC_COLUMNS)].round(6))


## Compare pooled confusion matrices

**What:** Plot both confusion matrices for the shared scored epochs on one color scale.

**Why:** Class totals and errors show where aggregate score changes come from.

In [ ]:
if report_ready:
    predictors = ("reference", "candidate")
    titles = ("Prepared reference", "Sensor-only replay")
    matrices = [pooled[key]["confusion_matrix"] for key in predictors]
    largest_count = max(count for matrix in matrices for row in matrix for count in row)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), layout="constrained")
    for ax, matrix, title in zip(axes, matrices, titles, strict=True):
        image = ax.imshow(matrix, cmap="Blues", vmin=0, vmax=largest_count)
        ax.set(title=title, xlabel="Predicted stage", ylabel="True stage")
        ax.set_xticks(range(4), STAGES, rotation=25, ha="right")
        ax.set_yticks(range(4), STAGES)
        for actual, row in enumerate(matrix):
            for predicted, count in enumerate(row):
                color = "white" if count > largest_count / 2 else "black"
                ax.text(predicted, actual, str(count), ha="center", va="center", color=color)
    fig.colorbar(image, ax=axes, label="Epochs")
    plt.show()


## Review prediction, history, and participant changes

**What:** Show class, probability, and history differences. Plot paired participant score changes.

**Why:** Aggregate scores can hide changes for individual participants. A changed history can also alter probabilities without changing the predicted class.
These counts describe development replay. They do not estimate final-test or clinical performance.


In [ ]:
if report_ready:
    change_rows = [
        {"fold": fold, **comparison["changes"]}
        for fold, comparison in comparisons.items()
    ]
    change_rows.append({"fold": "Pooled", **pooled["changes"]})
    display(pd.DataFrame(change_rows).set_index("fold")[[
        "class_disagreements", "max_abs_probability_difference", "history_disagreements",
    ]])

    participant_rows = [
        {"participant_id": person,
         "accuracy_delta": scores["candidate"]["accuracy"] - scores["reference"]["accuracy"],
         "macro_f1_delta": scores["candidate"]["macro_f1"] - scores["reference"]["macro_f1"]}
        for person, scores in pooled["participants"].items()
    ]
    participant_deltas = pd.DataFrame(participant_rows).set_index("participant_id").sort_index()
    display(participant_deltas.round(6))
    ax = participant_deltas.plot.bar(figsize=(13, 4))
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set(title="Paired participant score changes", xlabel="Participant",
           ylabel="Candidate minus reference")
    plt.tight_layout()
    plt.show()


## Attribute the changed predictions

**What:** Align the verified saved predictions and count corrections, new errors, and changes outside altered histories.

**Why:** This distinguishes a history-policy effect from a change in weights or numerical behavior elsewhere.
The extra candidate predictions remain unscored. This cell reads development outputs only and does not run inference.


In [ ]:
if report_ready:
    candidate_rows = pd.read_csv(study.output_dir / "candidate-predictions.csv",
                                 dtype={"participant_id": str})
    reference_rows = pd.read_csv(study.output_dir / "reference-predictions.csv",
                                 dtype={"participant_id": str})
    paired = candidate_rows.merge(reference_rows, on=["participant_id", "time"],
                                  suffixes=("_candidate", "_reference"), validate="one_to_one")
    changed_history = paired.history_epochs_candidate.ne(paired.history_epochs_reference)
    changed_class = paired.prediction_candidate.ne(paired.prediction_reference)
    was_correct = paired.prediction_reference.eq(paired.target)
    now_correct = paired.prediction_candidate.eq(paired.target)

    probability_columns = [f"probability_{stage}" for stage in range(4)]
    candidate_values = paired[[f"{name}_candidate" for name in probability_columns]].to_numpy()
    reference_values = paired[[f"{name}_reference" for name in probability_columns]].to_numpy()
    unchanged_maximum = abs(candidate_values[~changed_history] -
                            reference_values[~changed_history]).max(initial=0.0)
    display(pd.DataFrame([{
        "corrected_errors": int((~was_correct & now_correct).sum()),
        "new_errors": int((was_correct & ~now_correct).sum()),
        "wrong_to_other_wrong": int((~was_correct & ~now_correct & changed_class).sum()),
        "net_correct": int(now_correct.sum() - was_correct.sum()),
        "class_changes_outside_changed_history": int((changed_class & ~changed_history).sum()),
        "unchanged_history_max_probability_difference": float(unchanged_maximum),
    }]))


## Observed full replay · 2026-11-25

All five controls reproduced the original chronological probabilities exactly. Histories and reset reasons matched.
Weights and fitted transforms stayed unchanged, evaluation mode remained enabled, and no gradients were created.
The candidate produced 23,674 predictions. Paired scoring used 22,705 targets, with 969 extra predictions and three disclosed tail exclusions.

Sensor-only history changed 566 shared contexts and 177 class decisions. All other probabilities stayed identical.
Accuracy changed from 66.298% to 66.426%; pooled macro-F1 changed from 0.590391 to 0.593919.
NLL fell from 0.858437 to 0.854614. These are small development gains, driven mainly by fold 03.
Fold 02 lost macro-F1; folds 04 and 05 lost accuracy. REM F1 also fell slightly.

The change corrected 89 errors and introduced 60 errors. Another 28 changed decisions remained wrong.
The result supports a usable sensor-only policy, not a general accuracy guarantee or a new trained model.
See [the measured findings](../docs/experiments/sensor-only-replay-2026-11-25/README.md).


## Export a report-only HTML copy

**What:** Execute report cells in a fresh kernel and save the HTML output.

**Why:** Export skips setup, replay, and recursive export. The source notebook stays output-free.

In [ ]:
%cd "{ROOT}"
!uv run --locked --inexact --group gru --group notebook python scripts/execute_notebook.py \
    --notebook notebooks/45_sensor_only_frozen_replay.ipynb
if get_ipython().user_ns["_exit_code"] != 0:
    raise RuntimeError("Frozen sensor replay notebook export failed")
